# 🚀 Notebook do Professor (Demo) — Aula 05: Embeddings e busca semântica com ChromaDB

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 05/14 — Módulo 2: RAG**  
**⏱️ 1h40min**  
**🔢 nomic-embed-text · ChromaDB**  
**🔁 Andaime 45%**  

---

## 🎯 Objetivo da aula

Entender como texto vira número e como números permitem encontrar documentos por significado — não por palavras-chave. Ao final, o grupo tem uma coleção ChromaDB do domínio pronta para o pipeline RAG da Aula 06.

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz as soluções resolvidas dos exercícios da aula.

---

# 🔬 Código da aula — slide a slide

### Slide 08 — nomic-embed-text — setup no LangChain e Ollama

In [ ]:
!pip install langchain-ollama langchain-community chromadb -q

from langchain_ollama import OllamaEmbeddings
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Modelo de embedding — não é um LLM de geração de texto
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",   # 274M params, janela 8k tokens
)

# Gerar embedding de uma frase — retorna lista de floats
vetor = embeddings.embed_query("Como fazer feijoada?")
print(f"Dimensões: {len(vetor)}")   # → 768 floats
print(f"Primeiros 5: {vetor[:5]}")  # → [-0.024, 0.031, ...]
print(f"Tipo: {type(vetor[0])}")    # → <class 'float'>

# Embeddings de múltiplos documentos de uma vez
docs = ["Pizza com queijo", "Macarrão ao sugo", "Motor de carro"]
vetores = embeddings.embed_documents(docs)
print(f"Documentos: {len(vetores)}")   # → 3
print(f"Dims cada: {len(vetores[0])}")  # → 768

### Slide 09 — Calcular similaridade cosseno — o mecanismo por baixo

In [ ]:
import numpy as np

def similaridade_cosseno(v1: list, v2: list) -> float:
    """Calcula similaridade entre dois vetores de embedding."""
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Comparar pares de frases
frases = {
    "pizza":    "Pizza napolitana com mozarela",
    "lasanha":  "Lasanha com queijo e presunto",
    "futebol":  "Futebol é o esporte mais popular",
    "carro":    "Motor do carro faz barulho",
}
vetores = {k: embeddings.embed_query(v) for k, v in frases.items()}

query_vec = embeddings.embed_query("comida italiana")

for nome, vec in vetores.items():
    sim = similaridade_cosseno(query_vec, vec)
    print(f"{nome:10s}: {sim:.4f}")
# pizza:    0.8945  ← alta similaridade
# lasanha:  0.8712  ← alta similaridade
# futebol:  0.4321  ← baixa similaridade
# carro:    0.2104  ← muito baixa similaridade

### Slide 11 — ChromaDB — add, query e persist

In [ ]:
import chromadb
from chromadb import Settings

# Cliente persistente — salva no disco (não perde ao reiniciar)
client = chromadb.PersistentClient(path="/content/chroma_db")

# Criar ou recuperar uma coleção
colecao = client.get_or_create_collection(
    name="dominio_grupo",
    metadata={"hnsw:space": "cosine"},  # usar similaridade cosseno
)

# Adicionar documentos com embeddings pré-computados
textos = ["Pizza napolitana com mozarela", "Lasanha bolonhesa com ricota"]
vetores = embeddings.embed_documents(textos)

colecao.add(
    documents=textos,
    embeddings=vetores,
    ids=["doc_1", "doc_2"],
    metadatas=[{"categoria":"italiana"}, {"categoria":"italiana"}],
)

# Buscar os k documentos mais similares à query
query_vec = embeddings.embed_query("comida italiana")
resultados = colecao.query(
    query_embeddings=[query_vec],
    n_results=2,
)
print(resultados["documents"])   # → [["Pizza napolitana...", "Lasanha..."]]
print(resultados["distances"])   # → [[0.10, 0.13]] (distância, não similaridade)

### Slide 12 — Metadata filtering — filtrar antes de buscar

In [ ]:
# Adicionar documentos com metadados ricos
colecao.add(
    documents=[
        "Pizza margherita é originária de Nápoles",
        "Frango à parmegiana — clássico brasileiro",
        "Churrasco gaúcho com costela e linguiça",
        "Feijoada completa com laranja",
    ],
    embeddings=embeddings.embed_documents([...]),
    ids=["d1","d2","d3","d4"],
    metadatas=[
        {"culinaria":"italiana",   "pais":"italia"},
        {"culinaria":"brasileira",  "pais":"brasil"},
        {"culinaria":"brasileira",  "pais":"brasil"},
        {"culinaria":"brasileira",  "pais":"brasil"},
    ],
)

# Buscar SOMENTE culinária brasileira
res = colecao.query(
    query_embeddings=[embeddings.embed_query("prato com carne")],
    n_results=2,
    where={"culinaria": "brasileira"},  # filtro antes da busca
)
# → pizza excluída mesmo que tenha boa similaridade com "carne"
# → retorna somente churrasco e feijoada

# Filtro com operador AND
colecao.query(..., where={"$and": [{"pais":"brasil"}, {"culinaria":"brasileira"}]})

### Slide 13 — ChromaDB via LangChain — a integração declarativa

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document

embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Criar vector store com LangChain Documents
documentos = [
    Document(page_content="Pizza napolitana", metadata={"fonte":"livro_1"}),
    Document(page_content="Macarrão ao sugo",   metadata={"fonte":"livro_1"}),
    Document(page_content="Churrasco gaúcho",   metadata={"fonte":"livro_2"}),
]

# from_documents calcula embeddings e salva tudo automaticamente
db = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    persist_directory="/content/chroma_langchain",
)

# Busca direta por similaridade
docs = db.similarity_search("comida italiana", k=2)
for d in docs:
    print(d.page_content, "|", d.metadata)

# Converter em retriever para uso em chain (Aula 06)
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},         # recuperar top-3
)
# → este retriever vai direto na chain RAG da Aula 06

### Slide 16 — Persistência e recarga da coleção

In [ ]:
# ─── SESSÃO 1: criar e salvar ───────────────────────────────
db = Chroma.from_documents(
    documents=documentos,
    embedding=embeddings,
    persist_directory="/content/chroma_ckp02",  # salva automaticamente
)
print(f"Documentos salvos: {db._collection.count()}")

# ─── SESSÃO 2: recarregar sem recalcular embeddings ─────────
db_recarregado = Chroma(
    persist_directory="/content/chroma_ckp02",  # lê do disco
    embedding_function=embeddings,              # para gerar embeddings de queries
)
print(f"Documentos recuperados: {db_recarregado._collection.count()}")

# Adicionar documentos a uma coleção existente
novos_docs = [Document(page_content="Novo documento do domínio")]
db_recarregado.add_documents(novos_docs)
# Só os novos docs têm embeddings computados — os antigos já estão no disco

# No Google Colab: copiar a pasta para o Drive para persistência entre sessões
!cp -r /content/chroma_ckp02 /content/drive/MyDrive/chroma_ckp02

### Slide 17 — similarity_search_with_score — avaliar a qualidade da recuperação

In [ ]:
# Busca com scores (distância cosseno — menor = mais similar)
resultados = db.similarity_search_with_score(
    "comida italiana",
    k=5,
)

for doc, score in resultados:
    # score é distância (0 = idêntico, 1 = sem relação)
    similaridade = 1 - score  # converter para similaridade
    relevante    = "✅" if similaridade > 0.75 else "⚠️"
    print(f"{relevante} [{similaridade:.3f}] {doc.page_content[:60]}")

# Exemplo de saída:
# ✅ [0.943] Pizza margherita com tomate fresco e mozarela
# ✅ [0.921] Lasanha bolonhesa com ricota cremosa
# ✅ [0.897] Macarrão al dente com pesto de manjericão
# ⚠️ [0.623] Frango à parmegiana — clássico brasileiro
# ⚠️ [0.412] Tacos com carne temperada

# Filtrar por threshold de relevância
THRESHOLD = 0.75
relevantes = [doc for doc, score in resultados if (1-score) > THRESHOLD]

### Slide 22 — Python novo desta aula

In [ ]:
import numpy as np
import uuid

# 1. numpy — dot product e norma vetorial
a = np.array([0.8, 0.3, 0.5])
b = np.array([0.7, 0.4, 0.6])
np.dot(a, b)           # produto escalar (dot product)
np.linalg.norm(a)      # norma (magnitude) do vetor

# 2. Dict comprehension com zip — gerar vetores por chave
nomes   = ["pizza", "carro"]
vetores = [embeddings.embed_query(n) for n in nomes]
mapa    = {k: v for k, v in zip(nomes, vetores)}  # zip e dict comp

# 3. uuid — gerar IDs únicos para documentos
doc_id = str(uuid.uuid4())   # "f47ac10b-58cc-4372-a567-0e02b2c3d479"

# 4. List comprehension com unpacking de tupla
resultados = [(doc, score)]  # retorno do similarity_search_with_score
relevantes = [doc for doc, score in resultados if (1-score) > 0.75]

# 5. f-string com formatação de float
print(f"Score: {score:.3f}")   # 3 casas decimais
print(f"Texto: {texto[:60]}")  # cortar string em 60 chars

---

## 🏋️ Exercícios Resolvidos — versão professor (executar no Colab)

As quatro soluções prontas dos exercícios de fixação do notebook do aluno — rode em sala, uma a uma.


### Exercício 1 — Cliente de embeddings

**O que a solução demonstra:** o cliente de embeddings do semestre em execução — `OllamaEmbeddings(model="nomic-embed-text")`, o retorno de `embed_query` (768 floats) e o lote com `embed_documents`.

**Pontos a destacar na execução:** mostre as dimensões e o tipo `float` no console; rode duas vezes a mesma frase para mostrar que o vetor é determinístico. Reforce: embedding não é LLM de texto.


In [ ]:
# ── Solução do Exercício 1 — cliente de embeddings: texto → vetor ──
!pip install langchain-ollama langchain-community chromadb numpy -q

from langchain_ollama import OllamaEmbeddings
import numpy as np
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

# Lacuna 1 resolvida — o modelo de embedding do semestre
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Lacuna 2 resolvida — a frase entra como input e sai como lista de floats
vetor = embeddings.embed_query("Como fazer feijoada?")
print(f"Dimensões: {len(vetor)} · tipo de cada valor: {type(vetor[0]).__name__}")
print(f"Primeiros 5 valores: {vetor[:5]}")

# Lacuna 3 resolvida — vetores de vários documentos de uma vez
docs = ["Pizza com queijo", "Macarrão ao sugo", "Motor de carro"]
vetores = embeddings.embed_documents(docs)
print(f"Documentos: {len(vetores)} · dims de cada: {len(vetores[0])}")

# Ponto a destacar em sala: a mesma frase gera sempre o mesmo vetor
vetor_2 = embeddings.embed_query("Como fazer feijoada?")
diff = max(abs(a - b) for a, b in zip(vetor, vetor_2))
print(f"Diferença máxima entre 2 execuções: {diff:.2e} (≈ 0 → determinístico)")


### Exercício 2 — Similaridade cosseno com numpy

**O que a solução demonstra:** a fórmula completa do Slide 09 rodando em dois pares — o par semântico (comida × pizza) e o par sem relação (comida × motor).

**Pontos a destacar na execução:** leia a régua com o grupo (~1.0 / ~0.5 / ~0.0) e compare os dois valores no console — a diferença entre os pares é o argumento da busca semântica.


In [ ]:
# ── Solução do Exercício 2 — similaridade cosseno com numpy (Slide 09) ──
!pip install langchain-ollama langchain-community chromadb numpy -q

from langchain_ollama import OllamaEmbeddings
import numpy as np
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Fórmula completa — a mesma conta do Slide 09
def similaridade_cosseno(v1, v2) -> float:
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

vetor_query = embeddings.embed_query("comida italiana")
vetor_doc   = embeddings.embed_query("Pizza napolitana com mozarela")
vetor_fora  = embeddings.embed_query("Motor do carro faz barulho")

print(f"comida × pizza : {similaridade_cosseno(vetor_query, vetor_doc):.4f}")
print(f"comida × motor : {similaridade_cosseno(vetor_query, vetor_fora):.4f}")
# Régua de leitura: ~1.0 = mesmo significado · ~0.5 = relação parcial · ~0.0 = sem relação


### Exercício 3 — Metadata filtering no ChromaDB

**O que a solução demonstra:** as três buscas sobre a mesma coleção — sem filtro, com o filtro simples e com o composto `$and` — no mesmo formato do andaime do aluno.

**Pontos a destacar na execução:** a pizza some do resultado com filtro mesmo com boa similaridade — o `filter` restringe o espaço de busca ANTES do k-NN, não o resultado. Ajuste as chaves de metadado ao domínio do grupo.


In [ ]:
# ── Solução do Exercício 3 — metadata filtering: filtrar antes de buscar ──
!pip install langchain-ollama langchain-community chromadb -q

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Corpus do exemplo (domínio culinária) — o andaime do aluno usa o domínio do grupo
documentos = [
    Document(page_content="Pizza margherita é originária de Nápoles", metadata={"culinaria": "italiana", "pais": "italia"}),
    Document(page_content="Lasanha bolonhesa com ricota", metadata={"culinaria": "italiana", "pais": "italia"}),
    Document(page_content="Frango à parmegiana — clássico brasileiro", metadata={"culinaria": "brasileira", "pais": "brasil"}),
    Document(page_content="Churrasco gaúcho com costela e linguiça", metadata={"culinaria": "brasileira", "pais": "brasil"}),
    Document(page_content="Feijoada completa com laranja", metadata={"culinaria": "brasileira", "pais": "brasil"}),
]
db = Chroma.from_documents(
    documentos, embeddings,
    collection_name="ex03_filtro",
    collection_metadata={"hnsw:space": "cosine"},
)

query = "prato com carne"

# (1) sem filtro — o k-NN decide só pela similaridade
print("SEM filtro:", [d.page_content for d in db.similarity_search(query, k=3)])

# (2) filtro simples — restringe o espaço de busca ANTES do k-NN
print("\nCOM filtro:", [d.page_content for d in db.similarity_search(query, k=3, filter={"culinaria": "brasileira"})])

# (3) filtro composto $and — os dois metadados simultaneamente
print("\nCOM $and  :", [d.page_content for d in db.similarity_search(
    query, k=3, filter={"$and": [{"pais": "brasil"}, {"culinaria": "brasileira"}]})])
# A pizza some do resultado COM filtro mesmo com boa similaridade com "carne" —
# o filter restringe o espaço de busca, não o resultado.


### Exercício 4 — Corrija a leitura do score

**O que a solução demonstra:** o erro clássico do Slide 17 em execução — `score > 0.75` seleciona exatamente os documentos MENOS relacionados; a correção converte com `1 - score`.

**Pontos a destacar na execução:** rode as duas versões lado a lado e compare no console. Distância 0.10 é ótima (similaridade 0.90); 0.90 é péssima.


In [ ]:
# ── Solução do Exercício 4 — o score é DISTÂNCIA cosseno (menor = mais similar) ──
!pip install langchain-ollama langchain-community chromadb -q

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
embeddings = OllamaEmbeddings(model="nomic-embed-text")

documentos = [
    Document(page_content="Pizza margherita com tomate fresco e mozarela"),
    Document(page_content="Lasanha bolonhesa com ricota cremosa"),
    Document(page_content="Macarrão al dente com pesto de manjericão"),
    Document(page_content="Frango à parmegiana — clássico brasileiro"),
    Document(page_content="Tacos com carne temperada"),
]
db = Chroma.from_documents(
    documentos, embeddings,
    collection_name="ex04_score",
    collection_metadata={"hnsw:space": "cosine"},
)

resultados = db.similarity_search_with_score("comida italiana", k=5)

print("Versão ERRADA (score > 0.75):")
for doc, score in resultados:
    if score > 0.75:
        print(f"  ❌ [dist={score:.3f}] {doc.page_content[:60]}")

THRESHOLD = 0.75
print("\nVersão CORRIGIDA (1 - score > 0.75):")
for doc, score in resultados:
    similaridade = 1 - score
    relevante = "✅" if similaridade > THRESHOLD else "⚠️"
    print(f"  {relevante} [{similaridade:.3f}] {doc.page_content[:60]}")

relevantes = [doc for doc, score in resultados if (1 - score) > THRESHOLD]
print(f"\nDocumentos relevantes: {len(relevantes)}")


## 📚 Referências da aula

- Docs ChromaDB — Documentação oficial: collections, add, query, persist, metadata filtering. docs.trychroma.com
- Docs LangChain — OllamaEmbeddings e integração com ChromaDB. python.langchain.com/docs/integrations/vectorstores/chroma
- Modelo Nomic AI — nomic-embed-text: especificações, benchmarks e casos de uso. huggingface.co/nomic-ai/nomic-embed-text-v1
- Paper Mikolov, T. et al. — "Efficient Estimation of Word Representations in Vector Space." (Word2Vec, 2013) — a fundação conceitual dos embeddings modernos. arxiv.org/abs/1301.3781
- Livro Goodfellow, I.; Bengio, Y.; Courville, A. — Deep Learning. Pearson, 2017. Cap. 15: Representação distribuída — a base teórica dos embeddings e espaços vetoriais.

---

**→ Próxima Aula — Aula 06 · 14/09** — Pipeline RAG completo — load, split, embed, retrieve, generate
  
Conectar o retriever à chain LCEL. O LLM responde com base nos seus documentos.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*